# Week 3, day 1 (afternoon) — Worksheet 10 SOLUTIONS: sorting and ranking   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Questions 5 and 6 are the ones to re-read. The default tie rule produces a
rank of `1.5`, which is not a position any student can be in.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Sorting and ranking. Run this once.
import pandas as pd

# The deck's three students...
students = pd.DataFrame(
    {"Student": ["Sara", "Ahmed", "Lina"], "Marks": [85, 90, 75]}
)

# ...and the same three with a fourth who TIES with Ahmed.
tied = pd.DataFrame(
    {"Student": ["Sara", "Ahmed", "Lina", "Omar"], "Marks": [85, 90, 75, 90]}
)

sales = pd.read_csv("data/sales.csv")

print(students)
print()
print(tied)

PART A — sorting

### Question 1

Ascending -> Lina 75, Sara 85, Ahmed 90. Descending -> Ahmed, Sara, Lina.

`ascending=True` is the default. Matches the deck's slide exactly.

In [ ]:
print("ascending (default):")
print(students.sort_values("Marks"))
print()
print("descending:")
print(students.sort_values("Marks", ascending=False))

### Question 2

`students` is **unchanged** -> still `['Sara', 'Ahmed', 'Lina']`.

`sort_values` returns a sorted copy and leaves the original alone —
exactly like `drop` in worksheet 06.

The same trap applies and it is arguably worse here, because a sorted
printout is so convincing. You look at an ordered table on screen and
conclude the frame is ordered. It is not, and the next `.head()` you take
returns the original first rows.

In [ ]:
print(students)
print()
print("still in the original order:", list(students["Student"]))

### Question 3

Index after sorting -> `[2, 0, 1]`. After `reset_index(drop=True)` -> `[0, 1, 2]`.

Sorting moved the rows and carried their labels along, so the index is now
out of order — the exact situation from worksheet 04 Q3 where `.loc[1]` and
`.iloc[1]` stop agreeing.

`reset_index(drop=True)` renumbers to match the new order and discards the
old labels. Use it when the ordering itself is the meaning — a leaderboard,
a top-10 — and skip it when the original row identity still matters, because
`drop=True` throws that away permanently.

In [ ]:
s = students.sort_values("Marks")
print(s)
print("index after sorting:", list(s.index))
print()
print("with reset_index(drop=True):")
print(s.reset_index(drop=True))

# Reset it when the new ORDER is the meaningful thing -- a leaderboard,
# a top-10 table -- and the original row numbers no longer mean anything.

### Question 4

Sorted by `Region` ascending then `Sales` descending -> the Atlantic block first, `12450.48` at the top.

`ascending` takes a list, one entry per sort column, so you can mix
directions. Here: regions alphabetically, and within each region the
largest orders first.

The sort is stable, meaning rows that tie on every sort key keep their
relative order from before. That is what makes multi-column sorting
predictable, and it is why `sort_values("a").sort_values("b")` is not the
same as `sort_values(["b", "a"])`.

In [ ]:
out = sales.sort_values(["Region", "Sales"], ascending=[True, False])
print(out.head(8)[["Region", "Sales"]])
print()
print("region ascending, and within each region, sales descending")

PART B — ranking, and what a tie does

### Question 5

Three distinct marks -> ranks `2.0`, `1.0`, `3.0`, `dtype: float64`.

The ranks are correct and they are **floats**, with no tie anywhere in the
data.

`rank()` always returns `float64`, because the `average` method can
produce halves and the dtype has to be able to hold them. So a column of
ranks will print as `1.0, 2.0, 3.0` even when every value is a whole
number, and comparing one to the integer `1` works but exporting them
writes `1.0` into your CSV.

In [ ]:
r = students["Marks"].rank(ascending=False)
print(r)
print()
print("dtype:", r.dtype)

### Question 6

Ahmed and Omar both scored 90 and both rank **`1.5`**. -> Sara `3.0`, Lina `4.0`.

There is no such thing as one-and-a-half place. `1.5` is the *average* of
the two positions the tied pair would have occupied — 1st and 2nd — and
that is the default the deck never names.

It is a defensible statistical convention and a terrible answer to publish.
Nobody finished 1.5th. And notice Sara is `3.0`: the average method
consumes both positions, so third place is genuinely third even though
nobody was second.

If a rank is going in front of a human, choose the method explicitly.

In [ ]:
out = tied.copy()
out["Rank"] = out["Marks"].rank(ascending=False)
print(out)
print()
print("Ahmed and Omar both scored 90 and both rank", out.loc[1, "Rank"])

### Question 7

`average` `[3.0, 1.5, 4.0, 1.5]` · `min` `[3.0, 1.0, 4.0, 1.0]` · `max` `[3.0, 2.0, 4.0, 2.0]` · `dense` `[2.0, 1.0, 3.0, 1.0]` · `first` `[3.0, 1.0, 4.0, 2.0]`

Five answers to one question, all correct, all different.

- **`min`** is the sports convention: joint first, joint first, then third.
  Nobody is awarded second place.
- **`max`** gives the tied pair second, which reads as a penalty for tying.
- **`dense`** never skips a number, so Sara is `2.0` — useful for grading
  bands, wrong for a league table.
- **`first`** breaks the tie by row order. It produces clean integers and
  the winner is decided by whoever happened to be entered first, which is
  arbitrary and completely invisible in the output.
- **`average`** is the default.

The deck lists 'average, min, or dense' as things ties need and stops
there. Which one you want is a policy question, and `first` in particular
will silently manufacture a winner out of spreadsheet row order.

In [ ]:
for m in ("average", "min", "max", "dense", "first"):
    print("%-8s ->" % m, tied["Marks"].rank(ascending=False, method=m).tolist())
print()
print("names in order:", list(tied["Student"]))

# A league table uses "min": joint first, joint first, then third -- nobody
# is awarded second place. "dense" never skips a number. "first" breaks
# the tie by row order, which is arbitrary but produces whole numbers.

### Question 8

Sorted -> Ahmed and Omar on top. Ranked with `method="min"` -> both `1.0`, original row order preserved.

Sorting answers 'what order should I read these in' and destroys the
original arrangement. Ranking answers 'what position is each row in' and
leaves everything where it was.

The practical difference is that a rank is a *column*. It survives
filtering, joins to other tables, and can be grouped on. A sort is a view;
the moment you concatenate or merge, it is gone.

In [ ]:
print("sorted:")
print(tied.sort_values("Marks", ascending=False))
print()
out = tied.copy()
out["Rank"] = out["Marks"].rank(ascending=False, method="min")
print("ranked, original order kept:")
print(out)

# Sorting is for reading. Ranking is for keeping -- the rank travels with
# the row, so you can still join, filter or group on the original order.

### Question 9

Top three -> `10852 Atlantic 12450.48`, `293 Nunavut 10123.02`, `15622 West 8581.25`. -> orders at rank 1: `1`; duplicated rank values: `0`.

Zero duplicated ranks across 300 rows means every single `Sales` value in
this file is distinct — no two orders came to the same amount.

That is worth noticing rather than assuming. It means the tie rule made no
difference *on this data*, so all five methods from Q7 would have produced
identical output here. Code that ranks this file would pass every test and
still be a coin-flip the day two orders match — which, with 1,093 orders in
the full file and prices repeating across products, is a matter of time.

A clean result on a sample proves the code ran, not that the policy is
right.

In [ ]:
sales["Rank"] = sales["Sales"].rank(ascending=False, method="min")
top = sales.sort_values("Rank").head(3)
print(top[["OrderID", "Region", "Sales", "Rank"]])
print()
print("orders at rank 1:", (sales["Rank"] == 1).sum())
print("duplicated rank values:", sales["Rank"].duplicated().sum())

### Question 10

`sales.sort_values("Revenue")` -> **raises** `KeyError: 'Revenue'`.

`Revenue` is a perfectly reasonable name for what this file calls `Sales`,
and that is exactly why this happens. Every organisation has both words in
circulation for the same quantity.

The error is the good outcome. Compare it with worksheet 06 Q9's
`errors="ignore"`, where the same class of mistake — naming a column that
is not there — produced silence and no change. Here you find out
immediately.

`list(df.columns)` before you start is the cheapest way to avoid the whole
category, and it also catches the trailing spaces from worksheet 08 Q7.

In [ ]:
print(sales.sort_values("Revenue"))